In [1]:
import os
from pathlib import Path
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [2]:
project_dir = Path.cwd()

train_path = project_dir / "df_train.csv"
processed_test_path = project_dir / "df_test_processed.csv"
raw_test_path = project_dir / "df_test.csv"

train_df = pd.read_csv(train_path)

if processed_test_path.exists():
    test_df = pd.read_csv(processed_test_path)
else:
    test_df = pd.read_csv(raw_test_path)

date_cols = [
    "CloseDate",
    "ContractStatusChangeDate",
    "ListingContractDate",
    "PurchaseContractDate"
]

for col in date_cols:
    train_df = train_df.drop(columns=[col], errors="ignore")
    test_df = test_df.drop(columns=[col], errors="ignore")

feature_cols = [col for col in train_df.columns if col != "ClosePrice" and col in test_df.columns]
train_df = train_df[["ClosePrice", *feature_cols]].copy()
test_df = test_df[["ClosePrice", *feature_cols]].copy()

object_cols = [
    col for col in feature_cols
    if str(train_df[col].dtype).startswith("string")
    or str(test_df[col].dtype).startswith("string")
    or train_df[col].dtype == "object"
    or test_df[col].dtype == "object"
]

train_df = train_df.drop(columns=object_cols, errors="ignore")
test_df = test_df.drop(columns=object_cols, errors="ignore")

feature_cols = [col for col in feature_cols if col not in object_cols]

X_train = train_df.drop(columns=["ClosePrice"], errors="ignore")
y_train = train_df["ClosePrice"].astype(float)

X_test = test_df.drop(columns=["ClosePrice"], errors="ignore")
y_test = test_df["ClosePrice"].astype(float)

for col in X_train.columns:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce").fillna(0)
for col in X_test.columns:
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce").fillna(0)


In [3]:
# ordinary linear regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)
lr_r2 = r2_score(y_test, lr_predictions)

print(f"Linear Regression R^2 Score: {lr_r2:.4f}")

Linear Regression R^2 Score: -170.6507


In [4]:
#Ridge regression model (linear regression with L2 regularization)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
ridge_predictions = ridge_model.predict(X_test)
ridge_r2 = r2_score(y_test, ridge_predictions)
print(f"Ridge Regression R^2 Score: {ridge_r2:.4f}")

Ridge Regression R^2 Score: -168.6478


In [5]:
# lasso regression model (linear regression with L1 regularization)
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train, y_train)
lasso_predictions = lasso_model.predict(X_test)
lasso_r2 = r2_score(y_test, lasso_predictions)
print(f"Lasso Regression R^2 Score: {lasso_r2:.4f}")

Lasso Regression R^2 Score: -176.1922


c:\Users\saman\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.125703e+15, tolerance: 5.987e+12
  model = cd_fast.enet_coordinate_descent(
